# KVO_03 — Multisite leave-one-site-out U-Net

Four-site UAV generalization experiment. For each fold, **three complete mosaics are training sites and the fourth mosaic is never used for fitting or model selection**. Within the three training sites, spatial 100 m blocks provide train/validation crowns. The held-out site is evaluated only after the best validation checkpoint is fixed.

No full-mosaic probability rasters are produced here. Outputs are fold checkpoints, reusable patch caches, training histories, pixel metrics and crown metrics. Native raster GSD is retained in this first multisite experiment; GSD normalization can be tested separately if warranted.

In [ ]:
!pip -q install geopandas rasterio pyogrio segmentation-models-pytorch scikit-learn matplotlib tqdm
from pathlib import Path
import json, hashlib, random, shutil, time, warnings, gc
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from google.colab import drive
warnings.filterwarnings('ignore')
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; torch.backends.cudnn.benchmark=True
print('DEVICE:',DEVICE); print(torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')

In [ ]:
DRIVE_ROOT=Path('/content/drive'); MYDRIVE=DRIVE_ROOT/'MyDrive'
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT),force_remount=True)
ROOT=MYDRIVE/'Colab Notebooks'/'Musanga'; GEO=ROOT/'data'/'Geo_Ref'; AD=GEO/'Annotations'
OUT=ROOT/'KVO_work'/'KVO_03_LOSO'; OUT.mkdir(parents=True,exist_ok=True)
SITES={
 'MOS_1_2':{'raster':GEO/'MOS_1_2_kvo.tif','annotations':AD/'Tree_Annotations_MOS_1_2_kvo.shp'},
 'Yoko':{'raster':GEO/'Yoko_kvo_georef.tif','annotations':AD/'Tree_Annotations_Yoko_kvo_georef.shp'},
 'Baego':{'raster':GEO/'Baego_georef.tif','annotations':AD/'Tree_Annotation_Baego.shp'},
 'Yangambi':{'raster':GEO/'Yangambi_georef.tif','annotations':AD/'Tree_Annotation_Yangambi_georef.shp'}}
for s,d in SITES.items(): assert d['raster'].exists() and d['annotations'].exists(),s
PATCH=256; STRIDE=256; IGNORE=255; BLOCK_M=100.; VAL_FRAC=.20
MEAN=np.array([.485,.456,.406],np.float32)[:,None,None]; STD=np.array([.229,.224,.225],np.float32)[:,None,None]
print('OUT:',OUT)

## 1 — Audit labels and create spatial train/validation blocks

Every crown receives a stable site-local `crown_id`. For a given LOSO fold, the excluded site is entirely external test data. Crowns in each of the remaining sites are assigned to 100 m blocks; approximately 20% of blocks are validation blocks.

In [ ]:
ANNS={}; audit=[]
for site,d in SITES.items():
    with rasterio.open(d['raster']) as src: crs=src.crs; bounds=src.bounds; gsd=(abs(src.transform.a),abs(src.transform.e)); dtype=src.dtypes[0]
    assert dtype=='uint8',f'{site}: expected uint8 RGB, got {dtype}'
    g=gpd.read_file(d['annotations']); assert 'Class_ID' in g.columns
    if g.crs!=crs: g=g.to_crs(crs)
    g=g[g.geometry.notna() & ~g.geometry.is_empty].copy(); g['class_id_num']=pd.to_numeric(g.Class_ID,errors='coerce'); g=g[g.class_id_num.notna()].copy(); g['class_id_num']=g.class_id_num.astype(int)
    g['target']=(g.class_id_num==1).astype(np.uint8); g['crown_id']=np.arange(len(g),dtype=np.int64); c=g.geometry.centroid
    g['block_id']=np.floor((c.x-bounds.left)/BLOCK_M).astype(int).astype(str)+'_'+np.floor((c.y-bounds.bottom)/BLOCK_M).astype(int).astype(str)
    ANNS[site]=g; audit.append({'site':site,'n_crowns':len(g),'musanga':int(g.target.sum()),'other':int((1-g.target).sum()),'blocks':g.block_id.nunique(),'gsd_x':gsd[0],'gsd_y':gsd[1]})
audit_df=pd.DataFrame(audit); display(audit_df); audit_df.to_csv(OUT/'site_audit.csv',index=False)

def train_val_assignment(g,seed):
    blocks=g.block_id.unique().copy(); rng=np.random.default_rng(seed)
    for attempt in range(1000):
        b=blocks.copy(); rng.shuffle(b); nv=max(1,int(round(VAL_FRAC*len(b)))); vb=set(b[:nv]); sp=np.where(g.block_id.isin(vb),'val','train')
        if all(g.loc[sp==q,'target'].nunique()==2 for q in ['train','val']): return pd.Series(sp,index=g.index)
    raise RuntimeError('Could not make two-class train/val split')

## 2 — Reusable per-fold patch cache

Only labelled crown pixels contribute to loss. Patches that mix train and validation labels are excluded. Cache fingerprints include raster and annotation sizes plus fold settings, so corrected annotation files automatically generate a new cache. The cache is stored on Drive and reused on later sessions.

In [ ]:
CACHE=OUT/'cache'; CACHE.mkdir(exist_ok=True)
def build_fold_cache(heldout,force=False):
    train_sites=[s for s in SITES if s!=heldout]; split_ann={}
    for i,s in enumerate(train_sites):
        g=ANNS[s].copy(); g['split']=train_val_assignment(g,SEED+100*i+list(SITES).index(heldout)); split_ann[s]=g
    payload={'heldout':heldout,'patch':PATCH,'stride':STRIDE,'block_m':BLOCK_M,'val_frac':VAL_FRAC,
             'files':{s:(SITES[s]['raster'].stat().st_size,SITES[s]['annotations'].stat().st_size,int(SITES[s]['annotations'].stat().st_mtime)) for s in train_sites}}
    fp=hashlib.sha1(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:12]; path=CACHE/f'{heldout}_patches_{fp}.npz'
    if path.exists() and not force:
        print('Reusing cache:',path); d=np.load(path); return d['X'],d['Y'],d['S'],split_ann,path
    X=[]; Y=[]; SS=[]
    for site in train_sites:
        rp=SITES[site]['raster']; g=split_ann[site]
        with rasterio.open(rp) as src:
            H,W=src.height,src.width; tr=src.transform
            # Disk-backed temporary masks avoid allocating full masks in RAM.
            tmpdir=Path('/content')/'musanga_kvo03'; tmpdir.mkdir(exist_ok=True)
            mask_paths={}
            for sp in ['train','val']:
                mp=tmpdir/f'{heldout}_{site}_{sp}.tif'; prof=src.profile.copy(); prof.update(count=1,dtype='uint8',nodata=IGNORE,compress='lzw',tiled=True)
                with rasterio.open(mp,'w',**prof) as dst:
                    # initialize blockwise to IGNORE
                    for _,w in dst.block_windows(1): dst.write(np.full((int(w.height),int(w.width)),IGNORE,np.uint8),1,window=w)
                # rasterize in memory only for this site; delete immediately after extraction
                a=g[g.split==sp]; m=rasterize([(geom,int(y)) for geom,y in zip(a.geometry,a.target)],out_shape=(H,W),transform=tr,fill=IGNORE,dtype='uint8')
                with rasterio.open(mp,'r+') as dst: dst.write(m,1)
                del m; mask_paths[sp]=mp
            with rasterio.open(mask_paths['train']) as mt, rasterio.open(mask_paths['val']) as mv:
                for r0 in tqdm(range(0,H,PATCH),desc=f'{heldout}: cache {site}'):
                    for c0 in range(0,W,PATCH):
                        hh=min(PATCH,H-r0); ww=min(PATCH,W-c0); w=Window(c0,r0,ww,hh)
                        a=mt.read(1,window=w); b=mv.read(1,window=w); ht=np.any(a!=IGNORE); hv=np.any(b!=IGNORE)
                        if ht==hv: continue
                        m=a if ht else b; arr=src.read([1,2,3],window=w,boundless=True,fill_value=0,out_shape=(3,PATCH,PATCH))
                        mm=np.full((PATCH,PATCH),IGNORE,np.uint8); mm[:hh,:ww]=m
                        X.append(arr.astype(np.uint8)); Y.append(mm); SS.append(0 if ht else 1)
            for mp in mask_paths.values(): mp.unlink(missing_ok=True)
    X=np.stack(X); Y=np.stack(Y); SS=np.asarray(SS,np.uint8); np.savez_compressed(path,X=X,Y=Y,S=SS)
    print('Saved:',path,'train/val patches:',int((SS==0).sum()),int((SS==1).sum()))
    return X,Y,SS,split_ann,path

## 3 — Train one LOSO fold

Best checkpoint is selected **only by validation AP from the three training sites**. The held-out site is never consulted during training. Existing checkpoints/history are reused unless `FORCE_TRAIN=True`.

In [ ]:
class DS(Dataset):
    def __init__(self,X,Y,S,code,aug=False): self.X=X; self.Y=Y; self.idx=np.where(S==code)[0]; self.aug=aug
    def __len__(self): return len(self.idx)
    def __getitem__(self,j):
        i=self.idx[j]; x=self.X[i].astype(np.float32)/255.; y=self.Y[i].copy()
        if self.aug:
            if np.random.rand()<.5: x=x[:,:,::-1].copy(); y=y[:,::-1].copy()
            if np.random.rand()<.5: x=x[:,::-1,:].copy(); y=y[::-1,:].copy()
            k=np.random.randint(4); x=np.rot90(x,k,axes=(1,2)).copy(); y=np.rot90(y,k).copy(); x=np.clip(x*np.random.uniform(.9,1.1),0,1)
        return torch.from_numpy((x-MEAN)/STD),torch.from_numpy(y.astype(np.int64))

def pixel_eval(model,dl):
    yy=[]; pp=[]; model.eval()
    with torch.inference_mode():
        for x,y in dl:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            with torch.amp.autocast('cuda',enabled=(DEVICE=='cuda')): p=torch.sigmoid(model(x).squeeze(1))
            v=y!=IGNORE; yy.append(y[v].cpu().numpy()); pp.append(p[v].float().cpu().numpy())
    y=np.concatenate(yy); p=np.concatenate(pp); return y,p

def train_fold(heldout,force_cache=False,force_train=False):
    X,Y,S,split_ann,cache_path=build_fold_cache(heldout,force_cache)
    bs=16 if DEVICE=='cuda' else 4
    tr=DataLoader(DS(X,Y,S,0,True),batch_size=bs,shuffle=True,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
    va=DataLoader(DS(X,Y,S,1,False),batch_size=bs,shuffle=False,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
    fd=OUT/heldout; fd.mkdir(exist_ok=True); ck=fd/'best.pt'; hp=fd/'training_history.csv'
    model=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',in_channels=3,classes=1).to(DEVICE)
    if ck.exists() and hp.exists() and not force_train:
        print('Existing fold checkpoint — SKIPPING TRAINING:',heldout); model.load_state_dict(torch.load(ck,map_location=DEVICE)); return model,split_ann,cache_path
    opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4); scaler=torch.amp.GradScaler('cuda',enabled=DEVICE=='cuda'); best=-1; wait=0; hist=[]
    for ep in range(1,31):
        model.train(); losses=[]; t=time.time()
        for x,y in tr:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=DEVICE=='cuda'):
                z=model(x).squeeze(1); v=y!=IGNORE; loss=nn.functional.binary_cross_entropy_with_logits(z[v],y[v].float())
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
        yv,pv=pixel_eval(model,va); ap=average_precision_score(yv,pv); auc=roc_auc_score(yv,pv); hist.append({'epoch':ep,'loss':np.mean(losses),'val_auc':auc,'val_ap':ap,'seconds':time.time()-t})
        print(f'{heldout} ep {ep:02d} loss={np.mean(losses):.4f} val AUC={auc:.4f} AP={ap:.4f}')
        if ap>best: best=ap; wait=0; torch.save(model.state_dict(),ck)
        else:
            wait+=1
            if wait>=6: break
    pd.DataFrame(hist).to_csv(hp,index=False); model.load_state_dict(torch.load(ck,map_location=DEVICE)); return model,split_ann,cache_path

## 4 — Evaluate the completely unseen site

Evaluation predicts only crown bounding windows plus context; it does **not** generate a wall-to-wall raster. Metrics are threshold-free pixel ROC-AUC/AP and crown ROC-AUC/AP using mean probability. Crown summaries are persisted per fold.

In [ ]:
HALO=64
def predict_window(model,src,win):
    arr=src.read([1,2,3],window=win,boundless=True,fill_value=0).astype(np.float32)/255.; ph=(32-arr.shape[1]%32)%32; pw=(32-arr.shape[2]%32)%32
    arr=np.pad(arr,((0,0),(0,ph),(0,pw))); x=torch.from_numpy((arr-MEAN)/STD).unsqueeze(0).to(DEVICE)
    with torch.inference_mode(),torch.amp.autocast('cuda',enabled=DEVICE=='cuda'): p=torch.sigmoid(model(x).squeeze()).float().cpu().numpy()
    return p[:int(win.height),:int(win.width)]

def test_heldout(model,site):
    g=ANNS[site]; yy=[]; pp=[]; rows=[]
    with rasterio.open(SITES[site]['raster']) as src:
        for row in tqdm(g.itertuples(),total=len(g),desc=f'TEST {site}'):
            geom=row.geometry; w=rasterio.windows.from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths(); w=w.intersection(Window(0,0,src.width,src.height))
            if w.width<1 or w.height<1: continue
            ew=Window(w.col_off-HALO,w.row_off-HALO,w.width+2*HALO,w.height+2*HALO); pf=predict_window(model,src,ew); p=pf[HALO:HALO+int(w.height),HALO:HALO+int(w.width)]
            inside=rasterize([(geom,1)],out_shape=(int(w.height),int(w.width)),transform=src.window_transform(w),fill=0,dtype='uint8').astype(bool); v=p[inside]
            if not len(v): continue
            yy.append(np.full(len(v),row.target,np.uint8)); pp.append(v.astype(np.float32)); rows.append({'site':site,'crown_id':row.crown_id,'class_id':row.class_id_num,'truth':int(row.target),'n_pixels':len(v),'prob_mean':float(v.mean()),'prob_median':float(np.median(v)),'prob_p90':float(np.quantile(v,.9))})
    y=np.concatenate(yy); p=np.concatenate(pp); c=pd.DataFrame(rows)
    m={'heldout_site':site,'n_pixels':len(y),'n_crowns':len(c),'n_musanga':int(c.truth.sum()),'n_other':int((1-c.truth).sum()),'pixel_auc':roc_auc_score(y,p),'pixel_ap':average_precision_score(y,p),'crown_auc':roc_auc_score(c.truth,c.prob_mean),'crown_ap':average_precision_score(c.truth,c.prob_mean)}
    return m,c

## 5 — Run the four LOSO folds

Each completed fold is reusable. `RUN_FOLDS` lets you run one fold at a time if desired. Start with **Baego**: it directly tests whether training-site diversity fixes the weakest KVO_02 transfer.

In [ ]:
RUN_FOLDS=['Baego','Yoko','Yangambi','MOS_1_2']  # edit to e.g. ['Baego'] for one fold
FORCE_CACHE=False; FORCE_TRAIN=False; FORCE_TEST=False
all_metrics=[]
for heldout in RUN_FOLDS:
    fd=OUT/heldout; fd.mkdir(exist_ok=True); mp=fd/'heldout_metrics.json'; cp=fd/'heldout_crown_probabilities.csv'
    model,split_ann,cache_path=train_fold(heldout,FORCE_CACHE,FORCE_TRAIN)
    if mp.exists() and cp.exists() and not FORCE_TEST:
        print('Existing held-out evaluation — reusing:',heldout); m=json.loads(mp.read_text()); c=pd.read_csv(cp)
    else:
        m,c=test_heldout(model,heldout); cp_tmp=fd/'heldout_crown_probabilities.tmp.csv'; c.to_csv(cp_tmp,index=False); cp_tmp.replace(cp); mp.write_text(json.dumps(m,indent=2))
    all_metrics.append(m); print(json.dumps(m,indent=2))
    del model; gc.collect(); torch.cuda.empty_cache() if DEVICE=='cuda' else None
summary=pd.DataFrame(all_metrics); summary.to_csv(OUT/'LOSO_summary.csv',index=False); display(summary)

## 6 — Compare with the frozen MOS_1_2 baseline

KVO_02 external-transfer metrics are loaded when available. The key comparison is whether a model trained on three heterogeneous sites improves prediction of the fourth relative to the original MOS_1_2-only model.

In [ ]:
k2=ROOT/'KVO_work'/'KVO_02_external_validation'/'external_site_metrics.csv'
if k2.exists():
    baseline=pd.read_csv(k2).rename(columns={'site':'heldout_site','pixel_roc_auc':'baseline_pixel_auc','pixel_average_precision':'baseline_pixel_ap','crown_roc_auc_mean_probability':'baseline_crown_auc','crown_average_precision_mean_probability':'baseline_crown_ap'})
    cols=['heldout_site','baseline_pixel_auc','baseline_pixel_ap','baseline_crown_auc','baseline_crown_ap']; comparison=summary.merge(baseline[cols],on='heldout_site',how='left')
    display(comparison); comparison.to_csv(OUT/'LOSO_vs_MOS_baseline.csv',index=False)
else: print('KVO_02 metrics not found; LOSO summary is still complete.')
print('DONE — no full probability rasters were generated.')